In [0]:
%pip install python-dotenv

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

In [0]:
storage_account_name = "adlsgen2detraining2026"
container_name = "crypto-data-ronit"
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# --- 2. PATH DEFINITIONS ---
bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze/raw_market_data"
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"
silver_global_path = f"{silver_base_path}/global_metrics"

# Specific Silver Table Paths
silver_quotes_path = f"{silver_base_path}/crypto_quotes"

In [0]:
gold_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/gold"

# --- 2. DAILY PRICE SUMMARY & MOMENTUM ---
# Creating a table for day-to-day trends and price movements
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg, stddev, col

quotes_df = spark.read.format("delta").load(silver_quotes_path)

window_spec = Window.partitionBy("coin_id").orderBy("event_timestamp")

gold_daily_summary = quotes_df.withColumn("prev_day_price", lag("price_usd", 1).over(window_spec)) \
    .withColumn("daily_return", (col("price_usd") - col("prev_day_price")) / col("prev_day_price")) \
    .withColumn("7d_moving_avg", avg("price_usd").over(window_spec.rowsBetween(-7, 0))) \
    .select(
        "coin_id", "symbol", "price_usd", "daily_return", "7d_moving_avg", "event_timestamp"
    )

# --- 3. MARKET DOMINANCE & VOLATILITY ---
# Computing features required for the predictive component
gold_volatility_metrics = quotes_df.withColumn("rolling_std", stddev("price_usd").over(window_spec.rowsBetween(-14, 0))) \
    .select("coin_id", "symbol", "rolling_std", "market_cap", "event_timestamp")

# --- 4. WRITE TO GOLD ---
gold_daily_summary.write.format("delta").mode("overwrite").save(f"{gold_base_path}/daily_price_summary")
gold_volatility_metrics.write.format("delta").mode("overwrite").save(f"{gold_base_path}/volatility_metrics")

print("✅ Gold Layer Processing Complete!")

In [0]:
# --- 1. VALIDATE DAILY SUMMARY & MOMENTUM ---
print("Checking Daily Summary & 7-Day Moving Averages:")
gold_daily_df = spark.read.format("delta").load(f"{gold_base_path}/daily_price_summary")
display(gold_daily_df.sort(col("event_timestamp").desc()))

# --- 2. VALIDATE VOLATILITY & DOMINANCE ---
print("Checking Volatility Metrics & Market Cap:")
gold_vol_df = spark.read.format("delta").load(f"{gold_base_path}/volatility_metrics")
display(gold_vol_df.sort(col("event_timestamp").desc()))

In [0]:
# KPI: Total Crypto Market Cap Trend 
display(spark.read.format("delta").load(silver_global_path)
        .select("event_timestamp", "total_market_cap")
        .orderBy("event_timestamp"))

Databricks visualization. Run in Databricks to view.

In [0]:
# KPI: Top 5 Gainers (last 24 hours) 
from pyspark.sql.functions import col

display(gold_daily_df.select("symbol", "daily_return")
        .orderBy(col("daily_return").desc())
        .limit(5))

Databricks visualization. Run in Databricks to view.

In [0]:
# Feature Validation: Price vs Moving Average 
display(gold_daily_df.filter(col("symbol") == "BTC") # Filter for a specific coin
        .select("event_timestamp", "price_usd", "7d_moving_avg")
        .orderBy("event_timestamp"))

Databricks visualization. Run in Databricks to view.